<a href="https://colab.research.google.com/github/Polyanat59/Agentes_IA_Alura/blob/main/agente_alura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q groq pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 14.1 MB/s eta 0:00:00


In [2]:
import os
import re
import numpy as np

from pathlib import Path
from getpass import getpass
from pypdf import PdfReader
from groq import Groq
from sentence_transformers import SentenceTransformer
from IPython.display import Markdown, display


chave_groq = None

try:
    from google.colab import userdata

    chave_groq = (
        userdata.get("GROQ_API_KEY")
        or userdata.get("GROQ_API")
    )
except Exception:
    pass


if not chave_groq:
    chave_groq = getpass("Cole sua chave da Groq: ")


os.environ["GROQ_API_KEY"] = chave_groq
cliente_groq = Groq(api_key=chave_groq)

MODELO_IA = "llama-3.3-70b-versatile"

print("Groq configurada.")

Cole sua chave da Groq: ··········
Groq configurada.


In [7]:
PASTA_PDFS = Path("/content")

arquivos_pdf = sorted(PASTA_PDFS.glob("*.pdf"))


if not arquivos_pdf:
    try:
        from google.colab import files

        print("Selecione os documentos PDF que o agente deverá consultar.")
        files.upload()
        arquivos_pdf = sorted(PASTA_PDFS.glob("*.pdf"))

    except ImportError:
        PASTA_PDFS = Path(".")
        arquivos_pdf = sorted(PASTA_PDFS.glob("*.pdf"))


if not arquivos_pdf:
    raise FileNotFoundError(
        "Nenhum PDF foi encontrado."
    )


print(f"{len(arquivos_pdf)} PDF(s) encontrado(s):")

for arquivo in arquivos_pdf:
    print("-", arquivo.name)

2 PDF(s) encontrado(s):
- Manual de Fornecedores e Política de Compras — Mercado Central 24h.pdf
- Perguntas Frequentes (FAQ) — Clientes e Funcionários.pdf


In [8]:
def limpar_texto(texto):
    """Remove espaços e quebras excessivas do texto extraído."""

    texto = texto.replace("\x00", " ")
    texto = re.sub(r"\s+", " ", texto)

    return texto.strip()


paginas = []
pdfs_sem_texto = []


for arquivo in arquivos_pdf:
    leitor = PdfReader(str(arquivo))
    paginas_com_texto = 0

    for numero_pagina, pagina in enumerate(leitor.pages, start=1):
        texto = limpar_texto(pagina.extract_text() or "")

        if texto:
            paginas.append(
                {
                    "arquivo": arquivo.name,
                    "pagina": numero_pagina,
                    "texto": texto
                }
            )
            paginas_com_texto += 1

    if paginas_com_texto == 0:
        pdfs_sem_texto.append(arquivo.name)


print(f"{len(paginas)} página(s) com texto foram carregadas.")


if pdfs_sem_texto:
    print("\nAtenção: estes PDFs não possuem texto extraível:")
    for nome in pdfs_sem_texto:
        print("-", nome)

    print("Eles podem ser PDFs digitalizados e precisar de OCR.")

32 página(s) com texto foram carregadas.
